# Regression

Predicting a number, end to end. What we'll cover:

- The typical architecture of a regression network (and its hyperparameters)
- Input and output shapes (`X` = features, `y` = labels)
- Steps in modelling: create -> define loss/optimizer -> fit
- Improving a model and evaluating it with MAE / MSE
- Running experiments and comparing results
- Saving and loading a trained model


 📝 Core Concepts: Regression vs. Classification

When training neural networks, supervised machine learning problems are generally split into two main categories based on what the model is trying to predict:

*   **Regression (This Notebook):** Predicting a continuous **numerical value or number**. 
    *   *Examples:* Predicting house prices, temperature variations, stock values, or mapping a mathematical line (y = x + 10).
    *   *Core Question:* *"How much or how many?"*
    *   *Typical Output Layer:* A single neuron (`nn.Linear(in_features, out_features=1)`) without an activation function, outputting a raw scalar number.

*   **Classification:** Predicting a discrete **label, category, or class**.
    *   *Examples:* Identifying if an email is spam or not spam (binary), or sorting an image into a cat, dog, or bird (multi-class).
    *   *Core Question:* *"Which category does this belong to?"*
    *   *Typical Output Layer:* One or more neurons paired with a classification activation function (like Sigmoid or Softmax) to output probabilities.


## Typical architecture of a regression neural network

There is no single "correct" network, but this is the usual starting point for
"numbers in -> number out":

| Hyperparameter | Typical value |
| --- | --- |
| Input layer shape | same as number of features (1 here) |
| Hidden layer(s) | problem specific, minimum 1 |
| Neurons per hidden layer | generally 10 to 100 |
| Output layer shape | same as the prediction shape (1 here) |
| Hidden activation | usually ReLU |
| Output activation | none - regression outputs a raw number |
| Loss function | MSE or MAE (Huber if outliers) |
| Optimizer | SGD or Adam |

A **hyperparameter** is something you choose (number of layers, learning rate, epochs).
A **parameter** is something the model learns on its own (weights and biases).

We keep the layers linear in this notebook to stay focused on the workflow;
activation functions come up in later notebooks.


In [ ]:
# import
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
from torchinfo import summary # Modern replacement for tf.keras model.summary()

print(torch.__version__)


**What this cell does**
- Loads PyTorch, plotting, pandas and `torchinfo`.
- Prints the version so you can confirm your environment.


In [ ]:
# create features (converted to PyTorch Tensors)
x = torch.tensor([-7.0, -4.0, -1.0, 2.0, 5.0, 8.0, 11.0, 14.0])

# create labels
y = torch.tensor([3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0])

print(x.shape, y.shape)

# visualize
plt.scatter(x.numpy(), y.numpy())
plt.title("Housing Simulation: Features vs Price")
plt.xlabel("House Feature Layer Attribute (X)")
plt.ylabel("House Price Evaluation (Y)")
plt.grid(True)
plt.show()


**What this cell does**
- Creates 8 toy examples that follow `y = x + 10`.
- Plots them so you can see the straight-line pattern before training.


## Input and output shapes

The **input shape** is what goes into the model, the **output shape** is what comes out.
Mismatched shapes cause most of the confusing errors you'll hit, so check them early.

Our goal: use one `x` value to predict one `y` value.


In [ ]:
# scalar values, 1 dimention tensor
print("x,y (vector):", x, y)
print("x, y (vector) dimension:", x.dim(), y.dim())
print("x, y (vector) shape:", x.shape, y.shape)

# unsqueeze to add a dimension for the feature, convert to vector to metrix(2D tensor)
# Models want a feature dimension: (8,) -> (8, 1) = 8 samples, 1 feature(column)
print("\nx,y unsqueezed (metric):\nx=", x.unsqueeze(dim=1),", \ny=",y.unsqueeze(dim=1))
print("unsqueezed dimension:", x.unsqueeze(dim=1).dim(), y.unsqueeze(dim=1).dim())
print("unsqueezed shape:", x.unsqueeze(dim=1).shape, y.unsqueeze(dim=1).shape)

**What this section shows**
- `x` and `y` start as 1D tensors holding 8 values.
- `unsqueeze(dim=1)` turns them into `(8, 1)` = 8 samples, 1 feature.

So we have **1 feature -> 1 target**. That is why every model below starts with
`nn.Linear(in_features=1, ...)` and ends with `out_features=1`.

Remember the pattern is `y = x + 10`. If `x` was `17.0`, what should `y` be?
Keep that answer in mind - we'll ask the model the same question shortly.


## Steps in modelling

Every PyTorch model follows the same three steps:

1. **Create** the model - stack layers with `nn.Sequential` (or write a custom `nn.Module`).
2. **Define how it learns** - a loss function (`nn.L1Loss` is MAE) and an optimizer (`optim.SGD`).
3. **Fit** the model - PyTorch has no `.fit()`, so we write the loop ourselves:
   forward pass -> calculate loss -> zero gradients -> backward pass -> update weights.


In [ ]:
# 0. Set random seed for reproducibility
torch.manual_seed(42)

# Create features and labels (Ensure float32 and shape [8, 1] for PyTorch)
x = torch.tensor([-7.0, -4.0, -1.0, 2.0, 5.0, 8.0, 11.0, 14.0], dtype=torch.float32).unsqueeze(dim=1)
y = torch.tensor([3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0], dtype=torch.float32).unsqueeze(dim=1)

# 1. Create model using nn.Sequential
model = nn.Sequential(
    nn.Linear(in_features=1, out_features=1)
)

# 2. Define loss function (MAE) and Optimizer (SGD)
loss_fn = nn.L1Loss() # L1Loss is Mean Absolute Error (MAE)
# Note: TF's SGD default learning rate is 0.01. We use 0.001 here so the
# "undertrained" result is obvious; we raise it in the next step.
optimizer = optim.SGD(model.parameters(), lr=0.001)

# 3. Fit the model (PyTorch requires a manual training loop)
# One full-batch update per epoch (TF's fit() uses batches of 32 by default).
epochs = 5
for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    y_pred = model(x)

    # 2. Calculate loss
    loss = loss_fn(y_pred, y)

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Backward pass (backpropagation)
    loss.backward()

    # 5. Update weights
    optimizer.step()

    print(f"Epoch: {epoch+1} | Loss: {loss.item():.4f}")

# 4. Let's see model prediction again for 17.0
model.eval() # Put model in evaluation mode
with torch.inference_mode(): # Turns off gradient tracking for faster predictions
    test_x = torch.tensor([[17.0]], dtype=torch.float32)
    prediction = model(test_x)
    print(f"\nPrediction for 17.0: {prediction.item():.4f}")


**What this cell did**
- Trained the smallest model (1 input -> 1 output) for only 5 epochs with `lr=0.001`.
- The prediction is far from 27, so it needs more training.

It doesn't go very well... it should've output something close to 27-30.


## Improving the model

In [ ]:
# 2. Re-initialize Optimizer with a better Learning Rate
optimizer = optim.SGD(model.parameters(), lr=0.01) 

# 3. Fit the model for more epochs
epochs = 100 # Increased from 5 to 100
for epoch in range(epochs):
    model.train()
    
    y_pred = model(x)
    loss = loss_fn(y_pred, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1} | Loss: {loss.item():.4f}")

# 4. Predict again
model.eval()
with torch.inference_mode():
    test_x = torch.tensor([[17.0]], dtype=torch.float32)
    prediction = model(test_x)
    print(f"\nNew Prediction for 17.0: {prediction.item():.4f}")


**What this cell does**
- Retrains the same model with `lr=0.01` for 100 epochs instead of 5.
- The `x = 17` prediction moves from ~14 to ~30 (the target is 27).


## A bigger dataset

In [ ]:
# Make a bigger dataset (using torch.arange)
x = torch.arange(-100, 100, 4, dtype=torch.float32)
x

In [ ]:
y = torch.arange(-90, 110, 4, dtype=torch.float32)
y

In [ ]:
# Make label for data set
y = x + 10
y

In [ ]:
# visualise data
plt.scatter(x,y)

In [ ]:
len(x)

**What this cell does**
- Generates 50 points from -100 to 96 using the same `y = x + 10` rule.
- More data gives the model more examples to learn the pattern from.


## Splitting data: train / validation / test

Before training, hold some data back:

| Set | Purpose | Typical size |
| --- | --- | --- |
| Training | the model learns the pattern from this | 70-80% |
| Validation | tune hyperparameters while developing | 10-15% |
| Test | final check on data the model has never seen | 10-15% |

The point is **generalization**: performing well on unseen data. That only works
if the test set stays untouched until the final evaluation.

For simplicity we use train/test only here: the first 40 examples for training,
the last 10 for testing.


In [ ]:
# Reshape into a 2D matrix (Nx1) for PyTorch neural network compatability
x = x.unsqueeze(dim=1)
y = y.unsqueeze(dim=1)

# Split data to train and test sets (80% / 20%)
x_train = x[:40] 
y_train = y[:40]

x_test = x[40:]
y_test = y[40:]

print(f"Train size: {len(x_train)} | Test size: {len(x_test)}")

**What this cell does**
- Reshapes the data to `(N, 1)` and splits 80/20 into train and test.
- The 10 test points stay unseen until evaluation.


In [ ]:
# Visualize the training and test data
plt.figure(figsize=(10, 7))
# Convert tensors back to numpy arrays for plotting
plt.scatter(x_train.numpy(), y_train.numpy(), c="b", label="Training data")
plt.scatter(x_test.numpy(), y_test.numpy(), c="g", label="Testing data")
plt.legend()
plt.show()

**What this cell does**
- Plots training data in blue and test data in green.
- Confirms the split kept the same line pattern in both sets.


## Training the bigger model

In [ ]:
# A model with named layers (the PyTorch equivalent of TF's `name=` argument)
from collections import OrderedDict

my_model = nn.Sequential(OrderedDict([
    ('input_layer_myin_1', nn.Linear(in_features=1, out_features=50)),
    ('output_layer_myout_1', nn.Linear(in_features=50, out_features=1))
]))

print("--- my_model Summary ---")
summary(my_model, input_size=(1, 1))


### Reading a model summary

`summary()` lists the layers, their output shapes and the parameter counts:

- **Total params** - every learnable weight and bias in the model
- **Trainable params** - the ones the optimizer updates (all of them here)
- **Non-trainable params** - frozen parameters, common when reusing a pretrained model

Each `nn.Linear(in, out)` holds `in * out` weights plus `out` biases.
So `Linear(1, 50)` = 50 + 50 = 100 params, and `Linear(50, 1)` = 50 + 1 = 51.

> TF's `plot_model()` draws the model as a graph; in PyTorch that needs an extra
> library (`torchview` / `torchviz`). `summary()` is enough to debug shapes.


In [ ]:
# Exercise: more hidden units -> more parameters
# Linear(1, 10) = 10 + 10 = 20 params, Linear(10, 1) = 10 + 1 = 11 -> 31 total
exercise_model = nn.Sequential(
    nn.Linear(in_features=1, out_features=10),
    nn.Linear(in_features=10, out_features=1)
)
summary(exercise_model, input_size=(1, 1))


**What this cell does**
- Same 2-layer shape, but with 10 hidden units instead of 50.
- Prints 31 params - the `in * out + out` rule from the note above.


In [ ]:
# ==========================================
# COMPILE & FIT (Training My Model)
# ==========================================
loss_fn = nn.L1Loss() # MAE Loss
optimizer = optim.SGD(my_model.parameters(), lr=0.01)

epochs = 100
for epoch in range(epochs):
    my_model.train()
    
    # 1. Forward Pass
    y_pred_train = my_model(x_train)
    
    # 2. Calculate Loss
    loss = loss_fn(y_pred_train, y_train)
    
    # 3. Optimize Steps
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Verbose = 1 equivalent logging
    print(f"Epoch {epoch+1}/{epochs} - loss: {loss.item():.4f} - mae: {loss.item():.4f}")


**What this cell does**
- Trains the named 2-layer model for 100 epochs with MAE loss.
- Prints the loss every epoch (the `verbose=1` equivalent).


In [ ]:
# ==========================================
# EVALUATE / PREDICT
# ==========================================
my_model.eval()
with torch.inference_mode():
    # Make predictions on test data
    y_pred = my_model(x_test)


# ==========================================
# PLOTTING FUNCTION
# ==========================================
def plot_predictions(train_data=x_train,
                     train_labels=y_train,
                     test_data=x_test,
                     test_labels=y_test,
                     predictions=y_pred):
    plt.figure(figsize=(10, 7))
    
    # Detach tensors and map to numpy arrays for plotting
    plt.scatter(train_data.numpy(), train_labels.numpy(), c="b", label="Training data")
    plt.scatter(test_data.numpy(), test_labels.numpy(), c="g", label="Testing data")
    plt.scatter(test_data.numpy(), predictions.numpy(), c="r", label="Predictions")
    
    plt.legend()
    plt.show()

# Call the function to visualize results
plot_predictions()

**What this cell does**
- Predicts on the test set and plots predictions against ground truth.
- Defines `plot_predictions()` so later experiments can reuse it.


## Evaluating a model

The usual workflow is a loop: build -> evaluate -> tweak -> evaluate again.

"Visualize, visualize, visualize": the data, the model, the training, and the
predictions against the ground truth.

For regression, two metrics cover most cases:

- **MAE** (mean absolute error) - the average size of the miss, in the same units as `y`
- **MSE** (mean squared error) - squares the errors, so large misses hurt more

Lower is better for both.


### Shape mismatch: the classic bug

In TF, `predict()` returned `(10, 1)` while `y_test` was `(10,)`, so the MAE
function returned 10 values instead of one. In PyTorch both are `(10, 1)` here,
so it doesn't break - but mixing 1D and 2D tensors broadcasts silently:


In [ ]:
y_test_1d = y_test.squeeze()   # torch.Size([10])
y_pred_2d = my_model(x_test)   # torch.Size([10, 1])

# (10,) and (10, 1) broadcast to a (10, 10) matrix - wrong, and no error is raised!
(y_test_1d - y_pred_2d).shape


The fix is to make both sides the same shape before comparing: `squeeze()`
drops the extra dimension from both. The evaluation cell right below does that.


In [ ]:
# ==========================================
# 4. EVALUATE THE MODEL (Equivalent to model.evaluate)
# ==========================================
my_model.eval()
with torch.inference_mode():
    # Generate test predictions
    y_pred = my_model(x_test)
    
    # PyTorch alternative to model.evaluate() tracking MAE and MSE
    mae_loss_fn = torch.nn.L1Loss()
    mse_loss_fn = torch.nn.MSELoss()
    
    test_mae = mae_loss_fn(y_pred, y_test)
    test_mse = mse_loss_fn(y_pred, y_test)
    
    print(f"Test MAE: {test_mae.item():.4f}")
    print(f"Test MSE: {test_mse.item():.4f}")

# ==========================================
# CALCULATE METRICS DIRECTLY (With squeezing)
# ==========================================
# Just like tf.squeeze(), PyTorch uses torch.squeeze() to fix dimension mismatch issues
y_pred_squeezed = torch.squeeze(y_pred)
y_test_squeezed = torch.squeeze(y_test)

# Calculate MAE manually or using the functional API
mae = torch.mean(torch.abs(y_test_squeezed - y_pred_squeezed))
print(f"Direct Squeezed MAE calculation: {mae.item():.4f}")

# Calculate MSE manually
mse = torch.mean((y_test_squeezed - y_pred_squeezed) ** 2)
print(f"Direct Squeezed MSE calculation: {mse.item():.4f}")

**What this cell does**
- Calculates MAE and MSE two ways: loss functions and manual squeezed math.
- Both give the same numbers - a sanity check on the shape handling.


## Experiments and compare to improve a model

In [ ]:
# ==========================================
# 0. DEFINE METRIC FUNCTIONS & PLOTTING UNWRAPPERS
# ==========================================
def mae(y_true, y_pred):
    """Calculates Mean Absolute Error between squeezed targets."""
    return torch.mean(torch.abs(y_true.squeeze() - y_pred.squeeze())).item()

def mse(y_true, y_pred):
    """Calculates Mean Squared Error between squeezed targets."""
    return torch.mean((y_true.squeeze() - y_pred.squeeze()) ** 2).item()


In [ ]:
# Assuming x_train, y_train, x_test, y_test are already available as 2D tensors from previous blocks
# e.g., via unsqueeze(dim=1) which removes the need for tf.expand_dims

# ==========================================
# EXPERIMENT 1: model_1 (1 layer, 100 epochs)
# ==========================================
torch.manual_seed(42)


model_1 = nn.Sequential(
    nn.Linear(in_features=1, out_features=1)
)

loss_fn = nn.L1Loss()
optimizer_1 = optim.SGD(model_1.parameters(), lr=0.01)

# Fit model_1
for epoch in range(100):
    model_1.train()
    preds = model_1(x_train)
    loss = loss_fn(preds, y_train)
    optimizer_1.zero_grad()
    loss.backward()
    optimizer_1.step()

# Evaluate and predict
model_1.eval()
with torch.inference_mode():
    y_preds_1 = model_1(x_test)

plot_predictions(predictions=y_preds_1) # Using your existing plotting function

# Calculate metrics
mae_1 = mae(y_test, y_preds_1)
mse_1 = mse(y_test, y_preds_1)
print(f"Model 1 - MAE: {mae_1:.4f}, MSE: {mse_1:.4f}")


**What this cell does**
- Model 1: one linear layer, 100 epochs. This is the baseline.


In [ ]:
# ==========================================
# EXPERIMENT 2: model_2 (2 layers, 100 epochs)
# ==========================================
torch.manual_seed(42)

model_2 = nn.Sequential(
    nn.Linear(in_features=1, out_features=1),
    nn.Linear(in_features=1, out_features=1)
)

optimizer_2 = optim.SGD(model_2.parameters(), lr=0.01)

# Fit model_2 (verbose=0 equivalent by ignoring prints)
for epoch in range(100):
    model_2.train()
    preds = model_2(x_train)
    loss = loss_fn(preds, y_train)
    optimizer_2.zero_grad()
    loss.backward()
    optimizer_2.step()

# Evaluate and predict
model_2.eval()
with torch.inference_mode():
    y_preds_2 = model_2(x_test)

plot_predictions(predictions=y_preds_2)

# Calculate metrics
mae_2 = mae(y_test, y_preds_2)
mse_2 = mse(y_test, y_preds_2)
print(f"Model 2 - MAE: {mae_2:.4f}, MSE: {mse_2:.4f}")

**What this cell does**
- Model 2: two linear layers, 100 epochs. Tests whether depth helps.


In [ ]:
# ==========================================
# EXPERIMENT 3: model_3 (2 layers, 500 epochs)
# ==========================================
torch.manual_seed(42)

model_3 = nn.Sequential(
    nn.Linear(in_features=1, out_features=1),
    nn.Linear(in_features=1, out_features=1)
)

optimizer_3 = optim.SGD(model_3.parameters(), lr=0.01)

# Fit model_3 for 500 epochs
for epoch in range(500):
    model_3.train()
    preds = model_3(x_train)
    loss = loss_fn(preds, y_train)
    optimizer_3.zero_grad()
    loss.backward()
    optimizer_3.step()

# Evaluate and predict
model_3.eval()
with torch.inference_mode():
    y_preds_3 = model_3(x_test)

plot_predictions(predictions=y_preds_3)

# Calculate metrics
mae_3 = mae(y_test, y_preds_3)
mse_3 = mse(y_test, y_preds_3)
print(f"Model 3 - MAE: {mae_3:.4f}, MSE: {mse_3:.4f}")

**What this cell does**
- Model 3: same two layers, 500 epochs. Tests whether longer training helps.
- Result: it does worse - the overtrained case.


In [ ]:
# ==========================================
# COMPARE RESULTS VIA PANDAS DATAFRAME
# ==========================================
model_results = [["model_1", mae_1, mse_1],
                 ["model_2", mae_2, mse_2],
                 ["model_3", mae_3, mse_3]]

all_results = pd.DataFrame(model_results, columns=["model", "mae", "mse"])
all_results

From the experiments, `model_2` (2 layers, 100 epochs) performed best.

Notice that `model_3` trained longer but did worse - more epochs is not automatically
better. Training too long (or with too high a learning rate) can overshoot the good
solution, which is what we see here.

Each model you build is an experiment. Once the count grows, use a tracker instead of
printing and copying numbers:

- **TensorBoard** - `torch.utils.tensorboard.SummaryWriter` ships with PyTorch
- **Weights & Biases** - hosted experiment tracking with a similar workflow


## Saving and loading models

TF has two formats (SavedModel and HDF5). PyTorch also has two options:

1. **`state_dict()`** (recommended) - saves only the learned weights and biases.
   You recreate the same architecture before loading. This is the closest
   equivalent to a Keras weights/HDF5 file.
2. **The whole model object** - pickles architecture + weights together.
   Convenient, but it depends on your class definitions, so it breaks more easily.

Saving the weights is the safer habit for anything you'll ship: the checkpoint stays
independent of code changes. Always call `.eval()` on a loaded model before inference.


In [ ]:
import torch
import os

# Create a directory to store our models
os.makedirs("saved_models", exist_ok=True)

# ==========================================
# 1. Save using the standard PyTorch format (.pth / .pt)
# This saves ONLY the weights and biases (Highly Recommended)
# ==========================================
torch.save(obj=model_2.state_dict(), 
           f="saved_models/best_model_weights.pth")

# Check it out
print("Contents of saved_models directory:")
print(os.listdir("saved_models"))


# ==========================================
# 2. Save the ENTIRE model structure + weights
# (Closest equivalent to TensorFlow's HDF5/SavedModel approach)
# ==========================================
torch.save(obj=model_2, 
           f="saved_models/best_model_entire.pth")

# Check it out
print("\nVerifying entire model save:")
print(f"File exists: {os.path.exists('saved_models/best_model_entire.pth')}")


**What this cell does**
- Saves only the weights (`state_dict`) - the recommended option.
- Also saves the whole model object, for comparison.


In [ ]:
# load the ENTIRE model
# ==========================================
# APPROACH 1: Load Weights Only (State Dict)
# Highly recommended. Requires recreating the model blueprint.
# ==========================================

# 1. Recreate an empty instance of the model architecture blueprint
loaded_model_weights = nn.Sequential(
    nn.Linear(in_features=1, out_features=1),
    nn.Linear(in_features=1, out_features=1)
)

# 2. Inject the saved weights into the empty blueprint
loaded_model_weights.load_state_dict(torch.load("saved_models/best_model_weights.pth", weights_only=True))
loaded_model_weights.eval() # Set to evaluation mode

print("--- Approach 1: Loaded Weights Model Summary ---")
summary(loaded_model_weights, input_size=(40, 1))

**What this cell does**
- Rebuilds the same architecture, then loads the saved weights into it.
- This is the workflow you would use in a real application.


In [ ]:
# ==========================================
# APPROACH 2: Load the Entire Model Object
# Direct equivalent to tf.keras.models.load_model
# ==========================================

# Set weights_only=False because the file contains structural custom definitions
loaded_model_entire = torch.load("saved_models/best_model_entire.pth", weights_only=False)
loaded_model_entire.eval() # Set to evaluation mode

print("\n--- Approach 2: Loaded Entire Model Summary ---")
summary(loaded_model_entire, input_size=(40, 1))

**What this cell does**
- Loads the pickled model object directly - no rebuild needed.
- Convenient, but tied to the original class definitions.


In [ ]:
# ==========================================
# VERIFICATION & COMPARISON
# ==========================================
with torch.inference_mode():
    # 1. Generate predictions from the original active model instance
    model_2.eval()
    model_2_preds = model_2(x_test)
    
    # 2. Generate predictions from our newly loaded models
    weights_model_preds = loaded_model_weights(x_test)
    entire_model_preds = loaded_model_entire(x_test)

# Calculate MAEs using your custom mae() verification wrapper
mae_original = mae(y_test, model_2_preds)
mae_weights_loaded = mae(y_test, weights_model_preds)
mae_entire_loaded = mae(y_test, entire_model_preds)

# Output evaluation sanity checks
print("\n=== VERIFICATION RESULTS ===")
print(f"Original Model MAE:        {mae_original:.4f}")
print(f"Weights-Loaded Model MAE:  {mae_weights_loaded:.4f}")
print(f"Entire-Loaded Model MAE:   {mae_entire_loaded:.4f}")

print(f"\nDo original and weights-loaded model predictions match? {mae_original == mae_weights_loaded}")
print(f"Do original and entire-loaded model predictions match?  {mae_original == mae_entire_loaded}")

**What this cell does**
- Predicts with the original and both loaded models.
- All three MAEs match, so both save formats are faithful.


## A larger example

-  we're going to try predict the cost of medical insurance for individuals based on a number of different parameters such as, age, sex, bmi, children, smoking_status and residential_region.

To do, we'll leverage the pubically available Medical Cost dataset available from Kaggle and hosted on GitHub.